# Lab 1 - Practising Images as Numbers

**Student name:** ____________________  
**Student ID:** ____________________  
**Lab date:** ____________________

This practical applies Session 1: image arrays, pixel distance, alignment, signal and nuisance, intensity normalisation, and group-aware evaluation.

Run every provided code cell. Complete A1, B1, C1, D1, E1, and E2. Submit this executed notebook by **23:59 on the lab day**.


## Your notebook has two parts

**Part 1 is common.** Everyone builds and inspects the same small raw-pixel classifier.

**Part 2 is personal.** After you enter your student ID, the notebook creates a repeatable personal condition. Your brightness levels, shift, scenario code, group results, and sweep values differ from other students.

Use your own printed evidence in every written answer. Generic explanations without your scenario code and measured values are incomplete.


## Submission checklist

- Student name, ID, and lab date completed
- A1, B1, C1, D1, E1, and E2 completed
- Personal scenario code and measured values reported
- All code cells run with outputs and figures visible
- Exactly this completed notebook submitted by 23:59 on the lab day


## Setup


In [ ]:
import hashlib
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)
plt.rcParams["figure.dpi"] = 120

def show_image(image, title, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(3, 3))
    ax.imshow(np.clip(image, 0, 1), cmap="gray", vmin=0, vmax=1)
    ax.set_title(title)
    ax.axis("off")
    return ax

def show_grid(images, labels, title, columns=6):
    rows = int(np.ceil(len(images) / columns))
    fig, axes = plt.subplots(rows, columns, figsize=(2.1 * columns, 2.1 * rows))
    axes = np.atleast_1d(axes).ravel()
    for ax, image, label in zip(axes, images, labels):
        show_image(image, str(label), ax)
    for ax in axes[len(images):]:
        ax.axis("off")
    fig.suptitle(title, y=1.02)
    plt.tight_layout()
    plt.show()

def accuracy(predicted, expected):
    return 100 * np.mean(np.asarray(predicted) == np.asarray(expected))


## Part 1 - Common reference case


### Step 1 - Build a small labelled image set

Run the next cell. It creates six small images. Each class has a different spatial pattern, so location matters.


In [ ]:
def make_symbol(label, size=28):
    # The label-specific background is a deliberate shortcut in this toy dataset.
    # The spatial pattern remains the intended signal.
    background = 0.12 + 0.06 * label
    foreground = background + 0.18
    image = np.full((size, size), background, dtype=float)
    if label == 0:
        image[5:23, 5:9] = foreground
    elif label == 1:
        image[5:23, 19:23] = foreground
    elif label == 2:
        image[5:9, 5:23] = foreground
    elif label == 3:
        image[19:23, 5:23] = foreground
    elif label == 4:
        for i in range(5, 23):
            image[i, i] = foreground
            image[i, i + 1] = foreground
    elif label == 5:
        for i in range(5, 23):
            image[i, size - 1 - i] = foreground
            image[i, size - 2 - i] = foreground
    return image

train_labels = np.arange(6)
train_images = np.stack([make_symbol(label) for label in train_labels])
show_grid(train_images, [f"class {label}" for label in train_labels], "Common labelled templates")
print("train_images shape:", train_images.shape)
print("dtype:", train_images.dtype)
print("minimum and maximum:", (train_images.min(), train_images.max()))


### Your task A1 - Inspect one image

Complete and run the next cell. Do not hard-code the answer. Your function should work on any image array.


In [ ]:
def describe_image(image):
    # TODO: return a dictionary with shape, dtype, minimum, and maximum.
    # Hint: use image.shape, image.dtype, image.min(), and image.max().
    pass

description = describe_image(train_images[0])
print(description if description is not None else "Complete A1, then run this cell again.")


### Your written response A2

Using your output above, state the four properties you should check immediately after loading an image. Then state the shape of one template in this notebook.

_Write your answer here._


### Step 2 - Raw pixel template matching

Run the next cell. The classifier compares every query image with every stored template using mean absolute pixel difference.


In [ ]:
def mean_absolute_distance(image_a, image_b):
    return float(np.mean(np.abs(image_a - image_b)))

def predict_nearest(templates, labels, queries):
    distances = np.mean(
        np.abs(queries[:, None, :, :] - templates[None, :, :, :]),
        axis=(2, 3),
    )
    return labels[np.argmin(distances, axis=1)]

clean_predictions = predict_nearest(train_images, train_labels, train_images)
print("Clean raw-pixel accuracy:", f"{accuracy(clean_predictions, train_labels):.1f}%")
print("Predicted labels:", clean_predictions)


### Your written response A3

Why does this classifier get 100% here? State the alignment assumption that makes this result possible.

_Write your answer here._


## Part 2 - Your personal condition


### Step 3 - Enter your student ID and create your scenario

Replace DEMO-0000 with your own student ID, then run the cell. Do not change any other line. The notebook uses this ID only to create your repeatable scenario.


In [ ]:
student_id = "DEMO-0000"  # Replace with your own student ID before submission.

def stable_seed(value):
    digest = hashlib.sha256(("CV-LAB1-2026-" + value.strip()).encode("utf-8")).digest()
    return int.from_bytes(digest[:8], "big")

def build_profile(value):
    rng = np.random.default_rng(stable_seed(value))
    variant = int(rng.integers(0, 3))
    sweep_shift = tuple(int(x) for x in rng.choice([-1, 0, 1], size=2, replace=True))
    if variant == 0:
        title = "Two levels of dimming"
        group_a = {"alpha": float(rng.uniform(0.22, 0.42)), "shift": (0, 0)}
        group_b = {"alpha": float(rng.uniform(0.88, 0.98)), "shift": (0, 0)}
    elif variant == 1:
        title = "Dimming with a small alignment change"
        group_a = {"alpha": float(rng.uniform(0.26, 0.46)), "shift": tuple(int(x) for x in rng.choice([-1, 1], size=2))}
        group_b = {"alpha": float(rng.uniform(0.90, 0.98)), "shift": (0, 0)}
    else:
        title = "Brightening and clipping"
        group_a = {"alpha": float(rng.uniform(0.93, 1.07)), "shift": (0, 0)}
        group_b = {"alpha": float(rng.uniform(1.48, 1.90)), "shift": tuple(int(x) for x in rng.choice([-1, 0, 1], size=2))}
    alphas = np.round(np.sort([
        rng.uniform(0.20, 0.48),
        rng.uniform(0.64, 0.84),
        rng.uniform(0.88, 0.98),
        rng.uniform(1.03, 1.16),
        rng.uniform(1.35, 1.85),
    ]), 2)
    signature = hashlib.sha256(value.strip().encode("utf-8")).hexdigest()[:6].upper()
    return {
        "scenario_code": f"S1-{variant + 1}-{signature}",
        "scenario_name": title,
        "group_A": group_a,
        "group_B": group_b,
        "sweep_shift": sweep_shift,
        "sweep_alphas": alphas,
    }

profile = build_profile(student_id)
print("Scenario code:", profile["scenario_code"])
print("Scenario:", profile["scenario_name"])
print("Group A condition:", profile["group_A"])
print("Group B condition:", profile["group_B"])
print("Personal sweep alphas:", profile["sweep_alphas"])
if student_id == "DEMO-0000":
    print("Warning: replace DEMO-0000 with your own student ID before submission.")


### Step 4 - Build your personal test set and measure raw-pixel failure

Run the next cell. Both groups contain the same six object classes. They differ only in the listed nuisance condition.


In [ ]:
def shift_image(image, shift):
    dy, dx = shift
    return np.roll(np.roll(image, dy, axis=0), dx, axis=1)

def apply_condition(image, alpha, shift):
    shifted = shift_image(image, shift)
    return np.clip(shifted * alpha, 0, 1)

def condition_for_group(group_name):
    return profile["group_A"] if group_name == "A" else profile["group_B"]

test_images = []
test_labels_list = []
test_groups_list = []
for group_name in ["A", "B"]:
    condition = condition_for_group(group_name)
    for label, image in zip(train_labels, train_images):
        test_images.append(apply_condition(image, condition["alpha"], condition["shift"]))
        test_labels_list.append(label)
        test_groups_list.append(group_name)

test_images = np.stack(test_images)
test_labels = np.asarray(test_labels_list)
test_groups = np.asarray(test_groups_list)
raw_predictions = predict_nearest(train_images, train_labels, test_images)
raw_overall = accuracy(raw_predictions, test_labels)

show_grid(test_images, [f"{group}-{label}" for group, label in zip(test_groups, test_labels)], "Your personalised test images")
print(f"Raw-pixel overall accuracy: {raw_overall:.1f}%")
print("Raw predicted labels:", raw_predictions)


### Your task B1 - Group accuracy

Complete and run the next cell. Your function must calculate a separate percentage for every group present in test_groups.


In [ ]:
def group_accuracy(expected, predicted, groups):
    # TODO: return a dictionary mapping each group to its percentage accuracy.
    # Hint: for each value in np.unique(groups), make a Boolean mask.
    # Then use accuracy(predicted[mask], expected[mask]).
    pass

raw_group_scores = group_accuracy(test_labels, raw_predictions, test_groups)
print(raw_group_scores if raw_group_scores is not None else "Complete B1, then run this cell again.")


### Your written response B2

Write your scenario code. Then quote raw overall accuracy and the two group accuracies. Which group received the worse result, and what nuisance condition did that group have?

_Write your answer here._


## Part 3 - Representation and repair


### Your task C1 - Per-image intensity normalisation

Complete and run the next cell. Use the provided test output to check that a non-constant image has mean near 0 and standard deviation near 1.


In [ ]:
def normalise_one(image):
    # TODO: subtract the image mean and divide by image.std().
    # If the standard deviation is less than 1e-8, return np.zeros_like(image).
    pass

def normalise_batch(images):
    # TODO: apply normalise_one to every image and stack the results.
    pass

probe = normalise_one(train_images[0])
if probe is None:
    print("Complete C1, then run this cell again.")
else:
    print("mean after normalisation:", round(float(probe.mean()), 6))
    print("std after normalisation:", round(float(probe.std()), 6))


### Step 5 - Evaluate the repair

Run the next cell after completing B1 and C1. It evaluates exactly the same personalised test images as the raw-pixel baseline.


In [ ]:
probe = normalise_one(train_images[0])
if probe is None:
    print("Complete C1 before running the repair.")
else:
    repaired_predictions = predict_nearest(
        normalise_batch(train_images),
        train_labels,
        normalise_batch(test_images),
    )
    repaired_overall = accuracy(repaired_predictions, test_labels)
    repaired_group_scores = group_accuracy(test_labels, repaired_predictions, test_groups)
    print(f"Repaired overall accuracy: {repaired_overall:.1f}%")
    print("Repaired group scores:", repaired_group_scores)


### Your written response D1

Use your measured values to explain:
1. Which result improved after normalisation?
2. Did any failure remain?
3. Does your scenario include an alignment change or clipping? Explain why intensity normalisation can or cannot repair that part.

Include your scenario code and at least three numerical values from your output.

_Write your answer here._


## Part 4 - Your personalised stress test


### Your task E1 - Sweep the assigned intensity values

Complete and run the next cell. Use your own five alpha values shown in Step 3. The function must return a dictionary of raw-pixel accuracy values.


In [ ]:
def evaluate_personal_sweep(alphas):
    # TODO: for each alpha:
    # 1. create one transformed copy of every training image with sweep_shift,
    # 2. predict with the raw templates,
    # 3. store the accuracy in a dictionary keyed by alpha.
    # Return the dictionary.
    pass

sweep_scores = evaluate_personal_sweep(profile["sweep_alphas"])
print(sweep_scores if sweep_scores is not None else "Complete E1, then run this cell again.")


### Your written response E2

State:
- your scenario code,
- the alpha value with the lowest raw-pixel accuracy,
- the corresponding accuracy,
- whether this supports or weakens the idea that raw pixel distance is sensitive to nuisance variation.

Then add one sentence naming the signal and the nuisance in this lab.

_Write your answer here._


## Final submission check

1. Replace DEMO-0000 with your own student ID.
2. Restart and run all cells.
3. Confirm your scenario code appears and your personal figures and measurements remain visible.
4. Complete all answer areas and required code cells.
5. Submit this executed notebook by **23:59 on the lab day**.

Your ID creates a repeatable personal scenario. Another student's numbers should not be used as evidence for your submission.
